# Week 5 – ML Frameworks for Chooser Option Pricing

This notebook demonstrates the initial ML pipelines for:

- **Approach 1:** Volatility forecasting (RF/XGBoost) + BSM pricing.
- **Approach 2:** End-to-end supervised pricing (Linear / GBDT / MLP).

The goal is to wire up datasets, time-series splits, and basic models; full tuning and comparison will follow in later weeks.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.ml.datasets import (
    load_base_frame,
    build_volatility_dataset,
    build_volatility_multi_horizon_targets,
    build_pricing_dataset,
    build_volatility_sequence_dataset,
    time_series_split,
)
from src.ml.models_vol import train_rf_vol, train_xgb_vol, train_and_evaluate_vol_model
from src.ml.models_pricing import (
    train_linear_pricing,
    train_gbdt_pricing,
    train_and_evaluate_pricing_model,
)
from src.ml.metrics import regression_metrics

print("Project root:", PROJECT_ROOT)

In [ ]:
# Load base frame and build datasets
frame = load_base_frame("JPM")
print("Base frame shape:", frame.shape)

# Approach 1 dataset: volatility forecasting (main horizon 126)
X_vol, y_vol, dates_vol = build_volatility_dataset(frame, horizon_days=126)
(train_vol, val_vol, test_vol) = time_series_split(X_vol, y_vol, dates_vol)

# Additional multi-horizon targets for diagnostics
multi_h = build_volatility_multi_horizon_targets(frame, horizons=[21, 63, 126])

print("Volatility dataset:")
print("  Features:", list(X_vol.columns))
print("  X shape:", X_vol.shape)
print("  y shape:", y_vol.shape)
print("  Split sizes:", len(train_vol[1]), len(val_vol[1]), len(test_vol[1]))
print("  Multi-horizon label columns:", list(multi_h.columns))

# Optional LSTM sequence dataset (framework-ready)
X_seq, y_seq, dates_seq = build_volatility_sequence_dataset(frame, horizon_days=126, lookback_days=30)
print("\nLSTM sequence dataset:")
print("  X_seq shape:", X_seq.shape)
print("  y_seq shape:", y_seq.shape)

In [ ]:
# Volatility model demos (Approach 1)
X_train_v, y_train_v, _ = train_vol
X_val_v, y_val_v, _ = val_vol
X_test_v, y_test_v, _ = test_vol

rf_vol_model, rf_summary = train_and_evaluate_vol_model(
    train_rf_vol,
    X_train=X_train_v,
    y_train=y_train_v,
    X_val=X_val_v,
    y_val=y_val_v,
    X_test=X_test_v,
    y_test=y_test_v,
)

print("Random Forest vol metrics")
print("  val :", rf_summary["val"])
print("  test:", rf_summary["test"])

# XGBoost is optional depending on environment
try:
    xgb_vol_model, xgb_summary = train_and_evaluate_vol_model(
        train_xgb_vol,
        X_train=X_train_v,
        y_train=y_train_v,
        X_val=X_val_v,
        y_val=y_val_v,
        X_test=X_test_v,
        y_test=y_test_v,
    )
    print("\nXGBoost vol metrics")
    print("  val :", xgb_summary["val"])
    print("  test:", xgb_summary["test"])
except Exception as e:
    print("\nXGBoost skipped:", e)

In [ ]:
# Pricing dataset for Approach 2
K = 150.0
R_DEFAULT = 0.0015
Q = 0.0233
T1_YEARS = 0.5
T2_YEARS = 1.0
T1_DAYS = 126
T2_DAYS = 252

# Direct target: proxy actual PV; exclude bsm_price from features for fair direct modeling
X_price, y_price, dates_price, bsm_prices = build_pricing_dataset(
    frame,
    k=K,
    r_default=R_DEFAULT,
    q=Q,
    t1_years=T1_YEARS,
    t2_years=T2_YEARS,
    t1_days=T1_DAYS,
    t2_days=T2_DAYS,
    target_mode="direct",
    include_bsm_feature=False,
)

(train_price, val_price, test_price) = time_series_split(X_price, y_price, dates_price)

# Keep a date-indexed BSM series for baseline comparison on test
bsm_series = pd.Series(bsm_prices, index=dates_price)

print("Pricing dataset (direct target):")
print("  Features:", list(X_price.columns))
print("  X shape:", X_price.shape)
print("  y shape:", y_price.shape)
print("  Split sizes:", len(train_price[1]), len(val_price[1]), len(test_price[1]))

# Residual target variant (optional)
X_resid, y_resid, dates_resid, _ = build_pricing_dataset(
    frame,
    k=K,
    r_default=R_DEFAULT,
    q=Q,
    t1_years=T1_YEARS,
    t2_years=T2_YEARS,
    t1_days=T1_DAYS,
    t2_days=T2_DAYS,
    target_mode="residual",
    include_bsm_feature=True,
)
print("\nResidual dataset (optional) shape:", X_resid.shape)

In [ ]:
# Pricing model demos (Approach 2) + BSM baseline comparison
X_train_p, y_train_p, _ = train_price
X_val_p, y_val_p, _ = val_price
X_test_p, y_test_p, dates_test_p = test_price

# Baseline (BSM) predictions aligned to test dates
y_bsm_test = bsm_series.loc[dates_test_p].values
bsm_test_metrics = regression_metrics(y_test_p, y_bsm_test)

# Linear model (unified train/eval wrapper)
lin_model, lin_summary = train_and_evaluate_pricing_model(
    train_linear_pricing,
    X_train=X_train_p,
    y_train=y_train_p,
    X_val=X_val_p,
    y_val=y_val_p,
    X_test=X_test_p,
    y_test=y_test_p,
    y_baseline_test=y_bsm_test,
)

# GBDT model (unified train/eval wrapper)
gbdt_model, gbdt_summary = train_and_evaluate_pricing_model(
    train_gbdt_pricing,
    X_train=X_train_p,
    y_train=y_train_p,
    X_val=X_val_p,
    y_val=y_val_p,
    X_test=X_test_p,
    y_test=y_test_p,
    y_baseline_test=y_bsm_test,
)

summary = pd.DataFrame([
    {"model": "BSM baseline", **bsm_test_metrics},
    {"model": "Linear (test)", **{k: lin_summary["test"][k] for k in ["mae", "rmse", "r2"]}},
    {"model": "GBDT (test)", **{k: gbdt_summary["test"][k] for k in ["mae", "rmse", "r2"]}},
])

print("Validation metrics:")
print("  Linear:", lin_summary["val"])
print("  GBDT  :", gbdt_summary["val"])
print("\nTest comparison (vs BSM baseline):")
print(summary.to_string(index=False))

print("\nImprovement over BSM on test:")
print(f"  Linear MAE improvement: {lin_summary['test'].get('mae_improvement_pct', float('nan')):.2f}%")
print(f"  Linear RMSE improvement: {lin_summary['test'].get('rmse_improvement_pct', float('nan')):.2f}%")
print(f"  GBDT MAE improvement: {gbdt_summary['test'].get('mae_improvement_pct', float('nan')):.2f}%")
print(f"  GBDT RMSE improvement: {gbdt_summary['test'].get('rmse_improvement_pct', float('nan')):.2f}%")